# 03 · Feature Engineering

Turns the real, raw transaction rows into a model-ready, **leakage-safe** feature table.
Every feature is computed causally — using only information available strictly *before or
at* the moment the transaction is scored — so this pipeline is portable to a real-time
scoring service later without a train/serve skew.

Feature families:
1. **Velocity** — rolling transaction count/sum per card over 1h / 24h / 7d
2. **Amount anomaly** — z-score of this transaction vs. the card's own prior history
3. **Category novelty** — first time this card is used at this merchant category
4. **Geo** — real distance between the transaction and the cardholder's home address
   (see notebook 01/02 for why we use this instead of a fabricated "impossible travel"
   feature — merchant coordinates here aren't a continuous real path)
5. **Risk encodings** — smoothed, causal (expanding, pre-current-row) target encoding of
   merchant and category fraud rate
6. **Time** — cyclical hour-of-day / day-of-week, weekend flag, card tenure

**Deliberately excluded**: `gender`, `age`, `job`, `city_pop` — present in the raw data but
kept out of the model's feature set on fair-lending / disparate-impact grounds (see
notebook 01).

We then split the data **chronologically** (train / val / test by time) — the only valid
way to evaluate a fraud model.

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", 60)

txns = pd.read_parquet("../data/raw/transactions.parquet")
txns["timestamp"] = pd.to_datetime(txns["timestamp"])
txns = txns.sort_values("timestamp").reset_index(drop=True)
print(f"{len(txns):,} transactions, {txns.timestamp.min()} -> {txns.timestamp.max()}")

1,048,575 transactions, 2019-01-01 00:00:00 -> 2020-03-10 16:08:00


## Chronological train / val / test split

In [2]:
t_train_end = txns.timestamp.quantile(0.70)
t_val_end = txns.timestamp.quantile(0.85)

split = np.where(txns.timestamp <= t_train_end, "train",
         np.where(txns.timestamp <= t_val_end, "val", "test"))
txns["split"] = split

print(txns.groupby("split").agg(txns=("is_fraud", "size"), fraud_rate=("is_fraud", "mean")))
print("\nsplit date boundaries:")
print(f"  train ends: {t_train_end}")
print(f"  val ends:   {t_val_end}")
GLOBAL_RATE = txns.loc[txns.split == "train", "is_fraud"].mean()
print(f"\nTrain fraud rate used as smoothing prior: {GLOBAL_RATE:.4%}")

         txns  fraud_rate
split                    
test   157286    0.006059
train  734002    0.005925
val    157287    0.004476

split date boundaries:
  train ends: 2019-11-10 01:13:48
  val ends:   2019-12-24 07:48:00

Train fraud rate used as smoothing prior: 0.5925%


## Time features

In [3]:
hour = txns.timestamp.dt.hour
dow = txns.timestamp.dt.dayofweek

txns["hour_sin"] = np.sin(2 * np.pi * hour / 24)
txns["hour_cos"] = np.cos(2 * np.pi * hour / 24)
txns["dow_sin"] = np.sin(2 * np.pi * dow / 7)
txns["dow_cos"] = np.cos(2 * np.pi * dow / 7)
txns["is_weekend"] = (dow >= 5).astype(int)

## Card tenure

No account-signup date exists in this dataset, so we use "time since we first observed this card" as a tenure proxy — computed causally (`cummin`), so a card's very first transaction correctly gets tenure zero rather than leaking its full history length.

In [4]:
first_seen = txns.groupby("user_id")["timestamp"].cummin()
txns["card_tenure_days"] = (txns.timestamp - first_seen).dt.days

## Velocity features (rolling windows per card)

In [5]:
def rolling_velocity(d, window, label):
    s = d.set_index("timestamp").groupby("user_id")["amount"].rolling(window)
    out = s.agg(["count", "sum"])
    out.columns = [f"user_txn_count_{label}", f"user_amount_sum_{label}"]
    return out.reset_index(level=0, drop=True)

for window, label in [("1h", "1h"), ("24h", "24h"), ("7d", "7d")]:
    feat = rolling_velocity(txns[["user_id", "timestamp", "amount"]], window, label)
    txns[feat.columns] = feat.values

txns[[c for c in txns.columns if c.startswith("user_txn") or c.startswith("user_amount")]].describe()

,user_txn_count_1h,user_amount_sum_1h,user_txn_count_24h,user_amount_sum_24h,user_txn_count_7d,user_amount_sum_7d
count,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06
mean,1.195954e+00,8.518657e+01,5.050283e+00,3.570533e+02,2.711877e+01,1.897146e+03
std,4.704330e-01,1.945178e+02,3.267261e+00,4.772407e+02,1.507390e+01,1.491068e+03
min,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.030000e+00
25%,1.000000e+00,1.439000e+01,3.000000e+00,1.231900e+02,1.600000e+01,9.314100e+02
50%,1.000000e+00,5.446000e+01,4.000000e+00,2.425000e+02,2.500000e+01,1.531720e+03
75%,1.000000e+00,9.667000e+01,7.000000e+00,4.372400e+02,3.500000e+01,2.433765e+03
max,7.000000e+00,2.894890e+04,3.700000e+01,2.980930e+04,1.510000e+02,3.193753e+04


## Time since previous transaction (per card)

In [6]:
prev_ts = txns.groupby("user_id")["timestamp"].shift(1)
txns["time_since_prev_txn_sec"] = (txns.timestamp - prev_ts).dt.total_seconds()
txns["time_since_prev_txn_sec"] = txns["time_since_prev_txn_sec"].fillna(30 * 24 * 3600)

## Amount anomaly (expanding z-score vs. the card's own prior history)

In [7]:
g_user = txns.groupby("user_id")["amount"]
cum_n = g_user.cumcount()
cum_sum = g_user.cumsum() - txns["amount"]
cum_sumsq = (txns["amount"] ** 2).groupby(txns["user_id"]).cumsum() - txns["amount"] ** 2

prior_mean = cum_sum / cum_n.replace(0, np.nan)
prior_var = (cum_sumsq / cum_n.replace(0, np.nan)) - prior_mean ** 2
prior_std = np.sqrt(prior_var.clip(lower=1e-6))

txns["amount_zscore_user"] = ((txns["amount"] - prior_mean) / prior_std).fillna(0.0)
txns["amount_zscore_user"] = txns["amount_zscore_user"].replace([np.inf, -np.inf], 0.0)

## Category novelty

In [8]:
txns["is_new_category_for_user"] = ~txns.duplicated(subset=["user_id", "category"], keep="first")
txns["is_new_category_for_user"] = txns["is_new_category_for_user"].astype(int)
txns["is_new_category_for_user"].mean()

np.float64(0.012123119471663925)

## Distance from home

The real, well-grounded geo signal for this dataset (see notebook 02).

In [9]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

txns["distance_from_home_km"] = haversine_km(
    txns.home_lat, txns.home_lon, txns.merchant_lat, txns.merchant_lon
)

## Risk encodings: smoothed causal target encoding

`(prior_fraud_count + alpha * train_fraud_rate) / (prior_txn_count + alpha)`, using only
rows strictly before the current one in chronological order.

In [10]:
ALPHA = 20

def causal_rate_encoding(d, key, alpha, global_rate):
    g = d.groupby(key)["is_fraud"]
    prior_fraud = g.cumsum() - d["is_fraud"]
    prior_count = g.cumcount()
    return (prior_fraud + alpha * global_rate) / (prior_count + alpha)

txns["merchant_fraud_rate_prior"] = causal_rate_encoding(txns, "merchant_id", ALPHA, GLOBAL_RATE)
txns["category_fraud_rate_prior"] = causal_rate_encoding(txns, "category", ALPHA, GLOBAL_RATE)

txns[["merchant_fraud_rate_prior", "category_fraud_rate_prior"]].describe()

,merchant_fraud_rate_prior,category_fraud_rate_prior
count,1.048575e+06,1.048575e+06
mean,6.661400e-03,6.722917e-03
std,7.064825e-03,6.407463e-03
min,5.826010e-05,6.006135e-05
25%,1.621016e-03,2.315395e-03
50%,4.027145e-03,3.509088e-03
75%,9.874919e-03,8.181221e-03
max,8.405104e-02,3.146567e-02


## Assemble the model feature matrix

In [11]:
FEATURE_COLUMNS = [
    "amount", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend", "card_tenure_days",
    "user_txn_count_1h", "user_amount_sum_1h",
    "user_txn_count_24h", "user_amount_sum_24h",
    "user_txn_count_7d", "user_amount_sum_7d",
    "time_since_prev_txn_sec", "amount_zscore_user",
    "is_new_category_for_user", "distance_from_home_km",
    "merchant_fraud_rate_prior", "category_fraud_rate_prior",
]
ID_COLUMNS = ["transaction_id", "timestamp", "user_id", "merchant_id", "category", "split"]
LABEL_COLUMN = "is_fraud"

model_df = txns[ID_COLUMNS + FEATURE_COLUMNS + [LABEL_COLUMN]].copy()

assert model_df[FEATURE_COLUMNS].isnull().sum().sum() == 0, "unexpected nulls in feature matrix"
print(f"Feature matrix: {model_df.shape}, {len(FEATURE_COLUMNS)} features")
model_df.head()

Feature matrix: (1048575, 26), 19 features


,transaction_id,timestamp,user_id,merchant_id,category,split,amount,hour_sin,hour_cos,dow_sin,dow_cos,is_weekend,card_tenure_days,user_txn_count_1h,user_amount_sum_1h,user_txn_count_24h,user_amount_sum_24h,user_txn_count_7d,user_amount_sum_7d,time_since_prev_txn_sec,amount_zscore_user,is_new_category_for_user,distance_from_home_km,merchant_fraud_rate_prior,category_fraud_rate_prior,is_fraud
0,0b242abb623afc578575680df30655b9,2019-01-01 00:00:00,2703190000000000.0,"Rippin, Kub and Mann",misc_net,train,4.97,0.0,1.0,0.781831,0.62349,0,0,1.0,90.37,1.0,90.37,1.0,90.37,2592000.0,0.0,1,78.597568,0.005925,0.005925,0
1,1f76529f8574734946361c461b024d99,2019-01-01 00:00:00,630423000000.0,"Heller, Gutmann and Zieme",grocery_pos,train,107.23,0.0,1.0,0.781831,0.62349,0,0,1.0,108.09,2.0,198.46,2.0,198.46,2592000.0,0.0,1,30.212176,0.005925,0.005925,0
2,a1a22d70485983eac12b5b88dad1cf95,2019-01-01 00:00:00,38859500000000.0,Lind-Buckridge,entertainment,train,220.11,0.0,1.0,0.781831,0.62349,0,0,1.0,447.52,3.0,645.98,3.0,645.98,2592000.0,0.0,1,108.206083,0.005925,0.005925,0
3,6b849c168bdad6f867558c3793159a81,2019-01-01 00:01:00,3534090000000000.0,"Kutch, Hermiston and Farrell",gas_transport,train,45.00,0.0,1.0,0.781831,0.62349,0,0,1.0,73.17,2.0,520.69,4.0,719.15,2592000.0,0.0,1,95.673231,0.005925,0.005925,0
4,a41d7549acf90789359a9aa5346dcb46,2019-01-01 00:03:00,375534000000000.0,Keeling-Crist,misc_pos,train,41.96,0.0,1.0,0.781831,0.62349,0,0,1.0,47.55,3.0,568.24,5.0,766.70,2592000.0,0.0,1,77.556744,0.005925,0.005925,0


## Save processed splits

In [12]:
os.makedirs("../data/processed", exist_ok=True)

for name in ["train", "val", "test"]:
    subset = model_df[model_df.split == name].drop(columns=["split"])
    subset.to_parquet(f"../data/processed/{name}.parquet", index=False)
    print(f"{name}: {subset.shape}  fraud_rate={subset[LABEL_COLUMN].mean():.4%}")

import json
with open("../data/processed/feature_columns.json", "w") as f:
    json.dump({"features": FEATURE_COLUMNS, "label": LABEL_COLUMN}, f, indent=2)

print("\nSaved train/val/test parquet + feature_columns.json to data/processed/")

train: (734002, 25)  fraud_rate=0.5925%


val: (157287, 25)  fraud_rate=0.4476%


test: (157286, 25)  fraud_rate=0.6059%

Saved train/val/test parquet + feature_columns.json to data/processed/
